# VisionDrive: laptop vision → ESP32 instructions

The camera is the vehicle's forward view. The laptop combines **YOLO11n** object
detection with **Depth Anything V2 indoor metric Small**. Nearby surfaces can
affect steering without a known class name. Distance estimates and floor removal
are imperfect; this notebook does not establish physical driving accuracy.

The ESP32 receives expiring Wi-Fi commands; it does not run these models.
See the [system guide](../docs/laptop-esp32.md) and [midterm report](../docs/midterm-report.md).


In [ ]:
from pathlib import Path
import json, sys, subprocess
ROOT = Path.cwd().resolve()
if not (ROOT / 'demo.py').is_file() and (ROOT.parent / 'demo.py').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'demo.py').is_file(), 'Open the notebook from the repository or notebooks folder'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project:', ROOT.name)


## 1. Explicit setup

Use the configured laptop/evaluation environment and install requirements-laptop.txt
for CPU, or requirements-evaluation-gpu.txt for CUDA. Explicit setup downloads about
105 MB. Upstream laptop weights stay ignored. No ESP32 connection is automatic.


In [ ]:
INSTALL = False
DOWNLOAD_MODELS = False
if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-laptop.txt')], check=True)
if DOWNLOAD_MODELS:
    subprocess.run([sys.executable, str(ROOT/'scripts/setup_laptop_models.py')], check=True)
sources = ROOT/'weights/laptop/sources.json'
print(json.loads(sources.read_text()) if sources.exists() else 'Run model setup first')


## 2. Current evaluation dataset

The unchanged pretrained YOLO11n study has 400 fresh indoor photos: 80 development and
320 test, with source-verified person coverage. This is not a YOLO training
dataset. Historical ESPDet bottle/cup/box training remains separate in its notebook.
See the pretrained evaluation notebook and docs/dataset_card.md for provenance.


In [ ]:
from collections import Counter
records = [json.loads(line) for line in (ROOT/'data/public/indoor-evaluation-manifest.jsonl').read_text().splitlines()]
print('Images:', Counter(row['split'] for row in records))
print('Instances:', Counter(obj['label'] for row in records for obj in row['objects']))
assert len({row['sha256'] for row in records}) == len(records)


## 3. Inspect a real image, depth, detections, and instruction

Set `SAMPLE_IMAGE` to an image path on your computer. `DEPTH_SCALE` is an initial
camera-specific scale correction after comparing several predictions with tape
measurements. Predictions are estimated meters, never per-image min-max normalized
depth. A depth surface's internal score of 1 is a geometry flag, not probability.


In [ ]:
SAMPLE_IMAGE = None  # Example: ROOT / 'your-course-image.jpg'
DEPTH_SCALE = 1.0
if SAMPLE_IMAGE is not None:
    import cv2, time
    import matplotlib.pyplot as plt
    from visiondrive.config import Config
    from visiondrive.laptop_detector import LaptopDetector
    from visiondrive.decision import Controller
    from visiondrive.explain import render_explanation
    config = Config.load(ROOT / 'config/default.json')
    weights = ROOT / 'weights/laptop/yolo11n.pt'
    detector = LaptopDetector(weights, config,
                              ROOT/'weights/laptop/depth-indoor', depth_scale=DEPTH_SCALE)
    image = cv2.imread(str(SAMPLE_IMAGE))
    assert image is not None, 'Image could not be read'
    detector.warmup()
    started = time.monotonic()
    detections = detector.predict(image)
    finished = time.monotonic()
    controller = Controller(config)
    # Repeat this static observation for an illustration of stabilization only.
    # This does not count as independent temporal evidence for hardware driving.
    for i in range(config.stable_frames):
        decision = controller.update(detections, i, finished, finished)
    canvas = render_explanation(image, decision, config, detector.name,
        {'status':'running', 'metric_depth':True, 'inference_ms':(finished-started)*1000}, detector.label)
    depth = detector.depth.predict(image)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)); axes[0].axis('off')
    im = axes[1].imshow(depth, cmap='magma_r', vmin=0, vmax=5)
    axes[1].set_title('Estimated depth (meters)'); axes[1].axis('off')
    fig.colorbar(im, ax=axes[1]); plt.show()
    print('Floor model found:', detector.floor_found)
    print('Instruction:', decision.action.value, decision.reason)
    print([vars(d) for d in detections])
else:
    print('Choose SAMPLE_IMAGE to run real inference. No synthetic detections were substituted.')


## 4. Live vehicle-view demonstration

Run this from a terminal for the live view:

```powershell
.\run_logic_demo.cmd
```

This uses the current camera frame as the vehicle's forward view. Far objects do
not block; nearby middle obstacles cause a confirmed turn to a usable side.
Emergency clearance, invalid input, and unavailable routes produce STOP.

For the board, follow [Wi-Fi and receiver setup](../docs/laptop-esp32.md).
With no `--esp-host`, the application is visualization-only. Video/simulation
inputs cannot send motor commands. Physical hardware tests are still outstanding.


## 5. Midterm evidence and original training

- [Midterm requirement mapping and unseen-situation protocol](../docs/midterm-report.md)
- [Published dataset manifest](../data/public/bottle-cup-box-manifest.jsonl)
- [Original ESPDet training notebook](train_bottle_cup_box.ipynb)
- [Original held-out precision, recall, and mAP](../docs/model-report.md)
- [Current integration checks and limitations](../docs/verification.md)

The old detector's metrics do not describe the hybrid model. Integration tests
show that the pipeline works; they are not a new obstacle-avoidance accuracy score.
